# Silver:
## dbt Tests

In [0]:
%sql
SELECT COUNT(*) AS silver_rows
FROM nutrichain_lakehouse.silver.silver_openfood_products;

In [0]:
%sql
SELECT
    MIN(silver_processed_at) AS oldest_processed,
    MAX(silver_processed_at) AS newest_processed,
    COUNT(*) AS rows
FROM nutrichain_lakehouse.silver.silver_openfood_products;

In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT barcode) AS distinct_barcodes
FROM nutrichain_lakehouse.silver.silver_openfood_products;

In [0]:
WITH bronze_products AS (
    SELECT COUNT(*) AS bronze_pages
    FROM nutrichain_lakehouse.bronze.bronze_openfood_products_raw
),
silver AS (
    SELECT COUNT(*) AS silver_products
    FROM nutrichain_lakehouse.silver.silver_openfood_products
)
SELECT * FROM bronze_products, silver;

In [0]:
%sql
SELECT
    barcode,
    product_name,
    primary_country,
    country_iso_code,
    energy_kcal_per_100g,
    sugar_tier,
    silver_processed_at
FROM nutrichain_lakehouse.silver.silver_openfood_products
LIMIT 20;

In [0]:
%sql
SELECT
    country_iso_code,
    primary_country,
    COUNT(*) AS products
FROM nutrichain_lakehouse.silver.silver_openfood_products
GROUP BY 1, 2
ORDER BY products DESC
LIMIT 30;

## Transformation product num check

In [0]:
%sql
SELECT ingest_run_id, COUNT(*) AS products
FROM nutrichain_lakehouse.silver.silver_openfood_products
GROUP BY ingest_run_id 
ORDER BY ingest_run_id DESC;

## Column presence and types

In [0]:
%sql
DESCRIBE TABLE nutrichain_lakehouse.silver.silver_openfood_products;
SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT barcode) AS distinct_barcodes,
  SUM(CASE WHEN barcode IS NULL THEN 1 ELSE 0 END) AS null_barcode,
  SUM(CASE WHEN energy_kcal_per_100g IS NULL THEN 1 ELSE 0 END) AS null_kcal,
  SUM(CASE WHEN nutriscore_grade_mismatch THEN 1 ELSE 0 END) AS mismatch_rows
FROM nutrichain_lakehouse.silver.silver_openfood_products;

## Column quality summary

In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT barcode) AS distinct_barcodes,
    ROUND(100.0 * SUM(CASE WHEN energy_kcal_per_100g IS NULL THEN 1 ELSE 0 END) / COUNT(*), 2)
        AS missing_kcal_pct,
    SUM(CASE WHEN energy_kcal_per_100g IS NOT NULL
             AND (energy_kcal_per_100g < 0 OR energy_kcal_per_100g > 900) THEN 1 ELSE 0 END)
        AS kcal_out_of_range,
    SUM(CASE WHEN sugars_100g IS NOT NULL AND sugars_100g < 0 THEN 1 ELSE 0 END)
        AS negative_sugars,
    SUM(CASE WHEN sugar_tier IS NOT NULL
             AND LOWER(sugar_tier) NOT IN ('low', 'medium', 'high', 'unknown') THEN 1 ELSE 0 END)
        AS invalid_sugar_tier,
    SUM(CASE WHEN barcode IS NOT NULL
             AND NOT RLIKE(TRIM(barcode), '^[0-9]{8,14}$') THEN 1 ELSE 0 END)
        AS invalid_barcode_format,
    SUM(CASE WHEN nutriscore_grade_mismatch THEN 1 ELSE 0 END) AS nutriscore_mismatch_rows
FROM nutrichain_lakehouse.silver.silver_openfood_products;

## Outlier examples


In [0]:
%sql
SELECT barcode, product_name, energy_kcal_per_100g, fat_100g, sugars_100g, carbohydrates_100g
FROM nutrichain_lakehouse.silver.silver_openfood_products
WHERE ingest_run_id = 'YOUR_LATEST_RUN'
  AND (
    sugars_100g > carbohydrates_100g
    OR energy_kcal_per_100g > 700
  )
ORDER BY energy_kcal_per_100g DESC
LIMIT 30;

## Quality scorecard (one row)

In [0]:
%sql
SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT barcode) AS distinct_barcodes,
  SUM(CASE WHEN energy_kcal_per_100g IS NULL THEN 1 ELSE 0 END) AS null_kcal,
  SUM(CASE WHEN energy_kcal_per_100g < 0 OR energy_kcal_per_100g > 900 THEN 1 ELSE 0 END) AS kcal_out_of_range,
  SUM(CASE WHEN sugars_100g < 0 THEN 1 ELSE 0 END) AS negative_sugars,
  SUM(CASE WHEN sugars_100g > carbohydrates_100g THEN 1 ELSE 0 END) AS sugar_gt_carbs,
  SUM(CASE WHEN nutriscore_grade_mismatch THEN 1 ELSE 0 END) AS mismatch_rows,
  MIN(energy_kcal_per_100g) AS min_kcal,
  MAX(energy_kcal_per_100g) AS max_kcal
FROM nutrichain_lakehouse.silver.silver_openfood_products
WHERE ingest_run_id = 'YOUR_LATEST_RUN';

## Completeness for ONE run

In [0]:
%sql
SELECT
  COUNT(*) AS total_rows,
  ROUND(100.0 * SUM(CASE WHEN energy_kcal_per_100g IS NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_missing_kcal,
  ROUND(100.0 * SUM(CASE WHEN proteins_100g IS NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_missing_protein,
  ROUND(100.0 * SUM(CASE WHEN sugars_100g IS NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_missing_sugar,
  ROUND(100.0 * SUM(CASE WHEN nova_group IS NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_missing_nova,
  ROUND(100.0 * SUM(CASE WHEN nutriscore_grade_mismatch THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_mismatch
FROM nutrichain_lakehouse.silver.silver_openfood_products
--WHERE ingest_run_id = 'YOUR_LATEST_RUN';

## Column dictionary

In [0]:
DESCRIBE TABLE nutrichain_lakehouse.silver.silver_openfood_products;


## Rows per run

In [0]:
%sql
SELECT ingest_run_id, COUNT(*) AS products
FROM nutrichain_lakehouse.silver.silver_openfood_products
GROUP BY ingest_run_id
ORDER BY ingest_run_id DESC;